# What Is Perplexity? — Reproduce It Yourself

A companion notebook to the tutorial. Everything runs on a **free Colab GPU**
(`Runtime → Change runtime type → T4 GPU`). We measure real perplexity with
Qwen3, watch it word-by-word, and see two rules in action:
**(1)** a bigger model (same tokenizer) has lower perplexity, and
**(2)** a *different* tokenizer makes the numbers **not comparable**.

## Setup

In [ ]:
!pip -q install -U "transformers>=5.5" accelerate
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## The perplexity function

For each position: take the model's probability of the **true** next token,
turn it into a "surprise" (`-log p`), average, and exponentiate.

In [ ]:
def perplexity(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        logits = model(ids).logits
    logp = F.log_softmax(logits[0, :-1].float(), dim=-1)   # distribution at each position
    true_next = ids[0, 1:]                                  # words that actually came next
    surprise = -logp[range(len(true_next)), true_next]      # -log p of each true next word
    return torch.exp(surprise.mean()).item()

## Load Qwen3-0.6B

In [ ]:
name = "Qwen/Qwen3-0.6B"
tok06 = AutoTokenizer.from_pretrained(name)
m06 = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto").to(device).eval()

## Predictable text is low; gibberish is high

In [ ]:
sentences = {
    "very predictable": "The quick brown fox jumps over the lazy dog.",
    "ordinary English": "The cat sat on the warm windowsill in the afternoon sun.",
    "shuffled words":   "windowsill the on cat sun warm the afternoon sat in the.",
    "random words":     "purple xylophone quantum banana empire sang loudly yesterday backwards.",
}
for k, s in sentences.items():
    print(f"{k:18s}  ppl = {perplexity(m06, tok06, s):8.1f}")

## Word by word: where the surprise comes from

Watch the model be **confident** on function words ("on", "the") and
**surprised** by content words it couldn't guess.

In [ ]:
phrase = "The cat sat on the mat"
ids = tok06(phrase, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logp = F.log_softmax(m06(ids).logits[0, :-1].float(), dim=-1)
for i in range(ids.shape[1] - 1):
    nxt = ids[0, i + 1]
    p = logp[i, nxt].exp().item()
    ctx = tok06.decode(ids[0, :i + 1])
    print(f"after {ctx!r:34s} -> predict {tok06.decode([nxt])!r:8s}  p = {p:.3%}")
print("\nwhole-phrase perplexity:", round(perplexity(m06, tok06, phrase), 2))

## Rule 1 — a bigger model (same tokenizer) is less perplexed

Qwen3-1.7B shares Qwen3-0.6B's tokenizer, so their perplexities **are**
directly comparable. Bigger model, lower numbers.

In [ ]:
big = "Qwen/Qwen3-1.7B"
tokB = AutoTokenizer.from_pretrained(big)
mB = AutoModelForCausalLM.from_pretrained(big, torch_dtype="auto", device_map=device).eval()

print(f"{'':18s}   0.6B     1.7B")
for k, s in sentences.items():
    print(f"{k:18s}  {perplexity(m06, tok06, s):8.1f}  {perplexity(mB, tokB, s):8.1f}")

## Rule 2 — a *different* tokenizer is NOT comparable

Qwen3.5 is a strong model, but it uses a **different, larger tokenizer**
(~248k tokens vs Qwen3's ~152k). It splits text differently, which changes
the math — so its perplexity number **cannot** be compared to Qwen3's, even
though it's a capable model. Watch it score *higher* (worse-looking) than the
smaller Qwen3-0.6B — purely a tokenizer artifact. This is the caveat, live.

*(Qwen3.5-0.8B is a multimodal model; we load it text-only for perplexity.)*

In [ ]:
from transformers import AutoModelForImageTextToText
tok35 = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
m35 = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B", torch_dtype="auto", device_map=device).eval()

print("Qwen3 vocab:", tok06.vocab_size, " |  Qwen3.5 vocab:", tok35.vocab_size)
for k, s in sentences.items():
    print(f"{k:18s}  Qwen3.5-0.8B ppl = {perplexity(m35, tok35, s):8.1f}   (different tokenizer -> NOT comparable)")

## The twist — a third family (Gemma 4) and a second language (Japanese)

Two more tokenizer families and a new language. One subtlety first: some
tokenizers (Gemma's) define a **beginning-of-sequence (BOS)** token but don't
add it automatically — and the model was pretrained with it, so we must
prepend it for a fair reading. It's a **no-op for Qwen** (whose tokenizers
have no BOS), so the same helper is correct for everyone.

*Gemma 4 is a **gated** model: accept the license on its Hugging Face page and
run `from huggingface_hub import notebook_login; notebook_login()` first. It's
also multimodal — we load it text-only for perplexity.*

In [ ]:
from transformers import AutoModelForImageTextToText

def encode(tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    bos = tok.bos_token_id
    if bos is not None and (ids.shape[1] == 0 or ids[0, 0].item() != bos):
        ids = torch.cat([torch.tensor([[bos]]), ids], dim=1)   # Gemma needs it; Qwen has none
    return ids.to(device)

def ppl(model, tok, text):
    ids = encode(tok, text)
    with torch.no_grad():
        logp = F.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    tgt = ids[0, 1:]
    return torch.exp((-logp[range(len(tgt)), tgt]).mean()).item()

gemmas = {}
for name in ["google/gemma-4-E2B-it", "google/gemma-4-E4B-it"]:
    tokG = AutoTokenizer.from_pretrained(name)
    mG = AutoModelForImageTextToText.from_pretrained(name, torch_dtype="auto", device_map=device).eval()
    gemmas[name] = (mG, tokG)
    print(f"{name}  vocab={tokG.vocab_size}")
    for k, s in sentences.items():
        print(f"    {k:18s} ppl = {ppl(mG, tokG, s):9.1f}")

Bigger helps inside the Gemma family too (E4B < E2B) — but Gemma's
whole scale differs from Qwen's, because it's a third tokenizer. Now Japanese,
across all five models:

In [ ]:
models = {
    "Qwen3-0.6B": (m06, tok06), "Qwen3-1.7B": (mB, tokB), "Qwen3.5-0.8B": (m35, tok35),
    "Gemma-4-E2B": gemmas["google/gemma-4-E2B-it"], "Gemma-4-E4B": gemmas["google/gemma-4-E4B-it"],
}
ja = {
    "ordinary Japanese":    "猫が午後の日差しの中で暖かい窓辺に座っていた。",
    "predictable Japanese": "ありがとうございます。よろしくお願いします。",
}
for k, s in ja.items():
    print(k)
    for nm, (m, t) in models.items():
        print(f"    {nm:14s} ppl = {ppl(m, t, s):9.1f}")

## Why Japanese isn't comparable across families either

Same sentence, different splits. Count the tokens each family uses, then look
at how Qwen vs Gemma actually cut up the Japanese — Qwen's smaller vocabulary
even falls back to **raw UTF-8 bytes** (the `\ufffd` pieces) for a word it has
no whole token for.

In [ ]:
en, jp = sentences["ordinary English"], ja["ordinary Japanese"]
print(f"{'model':14s} {'EN tokens':>10s} {'JA tokens':>10s}")
for nm, (m, t) in models.items():
    print(f"{nm:14s} {len(t(en).input_ids):10d} {len(t(jp).input_ids):10d}")

print("\nHow the Japanese sentence is split:")
for nm in ["Qwen3-0.6B", "Gemma-4-E4B"]:
    m, t = models[nm]
    ids = t(jp).input_ids
    print(f"  {nm:12s} ({len(ids)}): {[t.decode([i]) for i in ids]}")

## Takeaways

- **Perplexity = exp(average −log p of the true next word).** Lower = less surprised.
- Surprise concentrates on **content words**; function words are cheap.
- **Same tokenizer, bigger model → lower perplexity** (a valid comparison) — holds for Qwen *and* Gemma.
- **Different tokenizer → the numbers are not comparable** — three families, three scales; even more so across languages, where tokenizers fragment text very differently.

That last rule is why, in the PHOTON conversion series, every comparison was
kept strictly within one tokenizer family.